# 07 - Visualizzazione dei circuiti quantistici

Questo notebook genera circuiti dimostrativi per BB84 ideale, BB84 con Eve intercept-resend, E91 ideale, CHSH e canali rumorosi rappresentativi.

I circuiti sono figure didattiche coerenti con gli esperimenti del progetto. Non introducono nuove simulazioni statistiche, non calcolano QBER, CHSH, key rate o altre metriche.

## Setup

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from math import pi

project_root = Path.cwd()
if not (project_root / "results").exists():
    project_root = project_root.parent

circuits_dir = project_root / "results" / "circuits"
circuits_dir.mkdir(parents=True, exist_ok=True)

print("Cartella circuiti pronta:", circuits_dir)

## Funzione di salvataggio

In [ ]:
def save_circuit(circuit, filename):
    # Funzione locale usata solo per salvare le figure dei circuiti.
    figure = circuit.draw("mpl")
    output_path = circuits_dir / filename
    figure.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Circuito salvato in:", output_path)

## BB84 ideale

Questi circuiti mostrano le quattro preparazioni base del protocollo BB84 e due round completi in base Z e X.

In [ ]:
q = QuantumRegister(1, "q")
c = ClassicalRegister(1, "c")

bb84_z_0 = QuantumCircuit(q, c)
bb84_z_0.barrier()
bb84_z_0.measure(q[0], c[0])
save_circuit(bb84_z_0, "bb84_prepare_z_0.png")

bb84_z_1 = QuantumCircuit(q, c)
bb84_z_1.x(q[0])
bb84_z_1.barrier()
bb84_z_1.measure(q[0], c[0])
save_circuit(bb84_z_1, "bb84_prepare_z_1.png")

bb84_x_plus = QuantumCircuit(q, c)
bb84_x_plus.h(q[0])
bb84_x_plus.barrier()
bb84_x_plus.measure(q[0], c[0])
save_circuit(bb84_x_plus, "bb84_prepare_x_plus.png")

bb84_x_minus = QuantumCircuit(q, c)
bb84_x_minus.x(q[0])
bb84_x_minus.h(q[0])
bb84_x_minus.barrier()
bb84_x_minus.measure(q[0], c[0])
save_circuit(bb84_x_minus, "bb84_prepare_x_minus.png")

In [ ]:
q = QuantumRegister(1, "q")
c = ClassicalRegister(1, "c")

bb84_round_x = QuantumCircuit(q, c)
bb84_round_x.h(q[0])
bb84_round_x.barrier()
bb84_round_x.h(q[0])
bb84_round_x.measure(q[0], c[0])
save_circuit(bb84_round_x, "bb84_round_x_basis.png")

bb84_round_z = QuantumCircuit(q, c)
bb84_round_z.x(q[0])
bb84_round_z.barrier()
bb84_round_z.measure(q[0], c[0])
save_circuit(bb84_round_z, "bb84_round_z_basis.png")

## BB84 con Eve intercept-resend

Il circuito seguente ? una rappresentazione didattica dell'attacco intercept-resend. Qiskit non rappresenta naturalmente la misura di un qubit e la preparazione fisica di un nuovo qubit nello stesso canale; per questo usiamo due qubit: uno viaggia da Alice a Eve, l'altro rappresenta il qubit reinviato da Eve a Bob.

In [ ]:
q = QuantumRegister(2, "q")
eve_c = ClassicalRegister(1, "eve")
bob_c = ClassicalRegister(1, "bob")

bb84_eve_demo = QuantumCircuit(q, eve_c, bob_c)

# Alice prepara |+> sul qubit q0.
bb84_eve_demo.h(q[0])
bb84_eve_demo.barrier()

# Eve misura q0 in base Z.
bb84_eve_demo.measure(q[0], eve_c[0])
bb84_eve_demo.barrier()

# Eve reinvia schematicamente un nuovo qubit q1 a Bob.
bb84_eve_demo.x(q[1])
bb84_eve_demo.barrier()

# Bob misura il qubit ricevuto.
bb84_eve_demo.measure(q[1], bob_c[0])

save_circuit(bb84_eve_demo, "bb84_intercept_resend_demo.png")

## E91 ideale

E91 usa una coppia entangled condivisa tra Alice e Bob. La preparazione dello stato di Bell |Phi+> si ottiene con H sul qubit di Alice e CNOT da Alice a Bob.

In [ ]:
q = QuantumRegister(2, "q")
c = ClassicalRegister(2, "c")

bell_prep = QuantumCircuit(q, c)
bell_prep.h(q[0])
bell_prep.cx(q[0], q[1])
save_circuit(bell_prep, "e91_bell_state_preparation.png")

measure_z_z = QuantumCircuit(q, c)
measure_z_z.h(q[0])
measure_z_z.cx(q[0], q[1])
measure_z_z.barrier()
measure_z_z.measure(q[0], c[0])
measure_z_z.measure(q[1], c[1])
save_circuit(measure_z_z, "e91_measurement_z_z.png")

measure_x_x = QuantumCircuit(q, c)
measure_x_x.h(q[0])
measure_x_x.cx(q[0], q[1])
measure_x_x.barrier()
measure_x_x.h(q[0])
measure_x_x.h(q[1])
measure_x_x.measure(q[0], c[0])
measure_x_x.measure(q[1], c[1])
save_circuit(measure_x_x, "e91_measurement_x_x.png")

## CHSH

Il test CHSH richiede pi? circuiti, uno per ogni coppia di impostazioni di misura. Il circuito seguente mostra un singolo setting rappresentativo. Il parametro S viene calcolato combinando le correlazioni ottenute da pi? setting.

In [ ]:
q = QuantumRegister(2, "q")
c = ClassicalRegister(2, "c")

theta_a = 0
theta_b = pi / 4

chsh_demo = QuantumCircuit(q, c)
chsh_demo.h(q[0])
chsh_demo.cx(q[0], q[1])
chsh_demo.barrier()
chsh_demo.ry(-theta_a, q[0])
chsh_demo.ry(-theta_b, q[1])
chsh_demo.measure(q[0], c[0])
chsh_demo.measure(q[1], c[1])

save_circuit(chsh_demo, "e91_chsh_measurement_demo.png")

## Rumore e decoerenza

Negli esperimenti del progetto i noise model sono applicati dal simulatore. I circuiti seguenti sono rappresentazioni schematiche utili per relazione e discussione orale.

Nel caso amplitude damping e Jaynes-Cummings, il circuito visuale rappresenta l'accoppiamento sistema-ambiente, mentre gli esperimenti numerici usano le funzioni implementate in `src`.

In [ ]:
q = QuantumRegister(1, "q")
c = ClassicalRegister(1, "c")

bb84_bit_flip_demo = QuantumCircuit(q, c)
bb84_bit_flip_demo.h(q[0])
bb84_bit_flip_demo.barrier()
bb84_bit_flip_demo.x(q[0])
bb84_bit_flip_demo.barrier()
bb84_bit_flip_demo.h(q[0])
bb84_bit_flip_demo.measure(q[0], c[0])
save_circuit(bb84_bit_flip_demo, "bb84_bit_flip_channel_demo.png")

In [ ]:
q = QuantumRegister(2, "q")
c = ClassicalRegister(1, "c")

amplitude_damping_demo = QuantumCircuit(q, c)

# q0 ? il sistema, q1 ? l'ambiente.
amplitude_damping_demo.x(q[0])
amplitude_damping_demo.barrier()
amplitude_damping_demo.cry(pi / 4, q[0], q[1])
amplitude_damping_demo.cx(q[1], q[0])
amplitude_damping_demo.barrier()
amplitude_damping_demo.measure(q[0], c[0])

save_circuit(amplitude_damping_demo, "bb84_amplitude_damping_demo.png")

In [ ]:
q = QuantumRegister(2, "q")
c = ClassicalRegister(2, "c")

e91_two_arm_noise = QuantumCircuit(q, c)
e91_two_arm_noise.h(q[0])
e91_two_arm_noise.cx(q[0], q[1])
e91_two_arm_noise.barrier()
e91_two_arm_noise.x(q[0])
e91_two_arm_noise.x(q[1])
e91_two_arm_noise.barrier()
e91_two_arm_noise.measure(q[0], c[0])
e91_two_arm_noise.measure(q[1], c[1])

save_circuit(e91_two_arm_noise, "e91_two_arm_noise_demo.png")

## Riepilogo figure

| Nome figura | Protocollo | Cosa rappresenta | Percorso |
|---|---|---|---|
| `bb84_prepare_z_0.png` | BB84 | Preparazione in base Z del bit 0 | `results/circuits/bb84_prepare_z_0.png` |
| `bb84_prepare_z_1.png` | BB84 | Preparazione in base Z del bit 1 | `results/circuits/bb84_prepare_z_1.png` |
| `bb84_prepare_x_plus.png` | BB84 | Preparazione dello stato |+> | `results/circuits/bb84_prepare_x_plus.png` |
| `bb84_prepare_x_minus.png` | BB84 | Preparazione dello stato |-> | `results/circuits/bb84_prepare_x_minus.png` |
| `bb84_round_x_basis.png` | BB84 | Round completo con misura in base X | `results/circuits/bb84_round_x_basis.png` |
| `bb84_round_z_basis.png` | BB84 | Round completo con misura in base Z | `results/circuits/bb84_round_z_basis.png` |
| `bb84_intercept_resend_demo.png` | BB84 + Eve | Schema didattico intercept-resend | `results/circuits/bb84_intercept_resend_demo.png` |
| `e91_bell_state_preparation.png` | E91 | Preparazione dello stato di Bell |Phi+> | `results/circuits/e91_bell_state_preparation.png` |
| `e91_measurement_z_z.png` | E91 | Misura di Alice e Bob in base Z | `results/circuits/e91_measurement_z_z.png` |
| `e91_measurement_x_x.png` | E91 | Misura di Alice e Bob in base X | `results/circuits/e91_measurement_x_x.png` |
| `e91_chsh_measurement_demo.png` | E91/CHSH | Un setting di misura CHSH | `results/circuits/e91_chsh_measurement_demo.png` |
| `bb84_bit_flip_channel_demo.png` | BB84 + rumore | Bit-flip schematico sul canale | `results/circuits/bb84_bit_flip_channel_demo.png` |
| `bb84_amplitude_damping_demo.png` | BB84 + rumore | Schema sistema-ambiente per damping | `results/circuits/bb84_amplitude_damping_demo.png` |
| `e91_two_arm_noise_demo.png` | E91 + rumore | Rumore schematico sui due bracci | `results/circuits/e91_two_arm_noise_demo.png` |